<a href="https://colab.research.google.com/github/hyeonggyeong-kim/comprehensive-project/blob/khg/ml/merge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================
# 난폭운전 데이터셋 전처리
# (기존 assign_driving_style 함수로 라벨 생성)
# =========================
# 코랩에 파일 직접 업로드


# =========================
# STEP 1. 데이터 불러오기
# =========================

df_agr = pd.read_excel("난폭운전_전용_대용량_데이터셋.xlsx")
print("데이터 크기:", df_agr.shape)

# =========================
# STEP 2. 컬럼명 영어로 변환
# =========================

df_agr = df_agr.rename(columns={
    "시간":                          "TIMESTAMP",
    "차량 속도 센서":                 "SPEED",
    "엔진 회전수":                    "ENGINE_RPM",
    "액셀러레이터 페달 위치 D":        "THROTTLE_POS",
    "엔진 부하":                      "ENGINE_LOAD",
    "흡기 매니폴드 절대 압력 (MAP)":   "INTAKE_MANIFOLD_PRESSURE",
    "흡입 공기 온도 (IAT)":            "AIR_INTAKE_TEMP",
    "엔진 냉각 온도":                  "ENGINE_COOLANT_TEMP",
    "속도_변화량":                    "SPEED_DIFF",
    "RPM_변화량":                     "RPM_DIFF",
    "페달_변화량":                    "THROTTLE_DIFF",
})

# =========================
# STEP 3. TIMESTAMP 변환
# =========================

def time_to_seconds(t):
    if isinstance(t, str):
        try:
            parts = t.strip().split(':')
            if len(parts) == 3:
                h, m, s = map(int, parts)
                return h * 3600 + m * 60 + s
        except:
            return np.nan
    return np.nan

df_agr["TIMESTAMP"] = df_agr["TIMESTAMP"].apply(time_to_seconds)

# =========================
# STEP 4. 이상치 처리
# =========================

if "SPEED" in df_agr.columns:
    df_agr.loc[df_agr["SPEED"] < 0, "SPEED"] = np.nan

if "ENGINE_RPM" in df_agr.columns:
    df_agr.loc[df_agr["ENGINE_RPM"] < 0, "ENGINE_RPM"] = np.nan

if "THROTTLE_POS" in df_agr.columns:
    df_agr.loc[
        (df_agr["THROTTLE_POS"] < 0) | (df_agr["THROTTLE_POS"] > 100),
        "THROTTLE_POS"
    ] = np.nan

# 결측치 중앙값 대체
from sklearn.impute import SimpleImputer
numeric_cols = df_agr.select_dtypes(include=[np.number]).columns.tolist()
imputer = SimpleImputer(strategy="median")
df_agr[numeric_cols] = imputer.fit_transform(df_agr[numeric_cols])

# =========================
# STEP 5. 이동평균 / 표준편차 생성
# =========================

window_size = 5

df_agr["SPEED_MA"]     = df_agr["SPEED"].rolling(window=window_size, min_periods=1).mean()
df_agr["SPEED_STD"]    = df_agr["SPEED"].rolling(window=window_size, min_periods=1).std().fillna(0)
df_agr["RPM_MA"]       = df_agr["ENGINE_RPM"].rolling(window=window_size, min_periods=1).mean()
df_agr["RPM_STD"]      = df_agr["ENGINE_RPM"].rolling(window=window_size, min_periods=1).std().fillna(0)
df_agr["THROTTLE_MA"]  = df_agr["THROTTLE_POS"].rolling(window=window_size, min_periods=1).mean()
df_agr["THROTTLE_STD"] = df_agr["THROTTLE_POS"].rolling(window=window_size, min_periods=1).std().fillna(0)

# =========================
# STEP 6. 이벤트 감지
# (기존과 동일한 임계값 사용)
# =========================

HARSH_ACCEL_THRESHOLD = 5
HARSH_BRAKE_THRESHOLD = -5
HIGH_RPM_THRESHOLD    = 2500
HIGH_SPEED_THRESHOLD  = 80

df_agr["HARSH_ACCEL"] = (df_agr["SPEED_DIFF"] > HARSH_ACCEL_THRESHOLD).astype(int)
df_agr["HARSH_BRAKE"] = (df_agr["SPEED_DIFF"] < HARSH_BRAKE_THRESHOLD).astype(int)
df_agr["HIGH_RPM"]    = (df_agr["ENGINE_RPM"] > HIGH_RPM_THRESHOLD).astype(int)
df_agr["HIGH_SPEED"]  = (df_agr["SPEED"] > HIGH_SPEED_THRESHOLD).astype(int)
df_agr["IDLING"]      = ((df_agr["SPEED"] == 0) & (df_agr["ENGINE_RPM"] > 700)).astype(int)

# =========================
# STEP 7. 라벨 생성 함수 정의 및 적용
# =========================

def assign_driving_style(row):
    score = 0

    if "HARSH_ACCEL" in row and row["HARSH_ACCEL"] == 1:
        score += 1

    if "HARSH_BRAKE" in row and row["HARSH_BRAKE"] == 1:
        score += 1

    if "HIGH_RPM" in row and row["HIGH_RPM"] == 1:
        score += 1

    if "HIGH_SPEED" in row and row["HIGH_SPEED"] == 1:
        score += 1

    if "SPEED_STD" in row and row["SPEED_STD"] > 8:
        score += 1

    if score >= 2:
        return "aggressive"
    elif score >= 1:
        return "normal"
    else:
        return "safe"

df_agr["DRIVING_STYLE"] = df_agr.apply(assign_driving_style, axis=1)

print("\n난폭운전 데이터 라벨 분포 (우리 기준):")
print(df_agr["DRIVING_STYLE"].value_counts())
print("\n비율:")
print(df_agr["DRIVING_STYLE"].value_counts(normalize=True).round(3))

# =========================
# STEP 8. 불필요한 컬럼 제거
# =========================

drop_cols = [
    "Lat.", "Lon.",
    "스로틀 위치 절대값",
    "난폭운전_라벨",
    "난폭운전_유형"
]
df_agr = df_agr.drop(columns=[c for c in drop_cols if c in df_agr.columns])

# =========================
# STEP 9. 기존 병합 데이터와 합치기
# =========================

df_existing = pd.read_csv("obd_merged_labeled.csv")

df_agr["DATA_SOURCE"]      = "aggressive_dataset"

# 공통 컬럼 기준 병합
merge_cols = [c for c in df_existing.columns if c in df_agr.columns]
print("\n병합 기준 컬럼:", merge_cols)

df_final = pd.concat(
    [df_existing[merge_cols], df_agr[merge_cols]],
    ignore_index=True
)

print("\n최종 병합 결과:")
print(f"  기존 데이터:       {len(df_existing)}건")
print(f"  난폭운전 데이터:   {len(df_agr)}건")
print(f"  합계:              {len(df_final)}건")
print(f"\n전체 라벨 분포:")
print(df_final["DRIVING_STYLE"].value_counts())
print("\n비율:")
print(df_final["DRIVING_STYLE"].value_counts(normalize=True).round(3))

# =========================
# STEP 10. 저장
# =========================

df_final.to_csv("obd_final.csv", index=False)
print("\n저장 완료: obd_final.csv")

데이터 크기: (6000, 16)

난폭운전 데이터 라벨 분포 (우리 기준):
DRIVING_STYLE
safe          3090
aggressive    2104
normal         806
Name: count, dtype: int64

비율:
DRIVING_STYLE
safe          0.515
aggressive    0.351
normal        0.134
Name: proportion, dtype: float64

병합 기준 컬럼: ['TIMESTAMP', 'SPEED', 'ENGINE_RPM', 'THROTTLE_POS', 'ENGINE_LOAD', 'ENGINE_COOLANT_TEMP', 'AIR_INTAKE_TEMP', 'SPEED_DIFF', 'RPM_DIFF', 'THROTTLE_DIFF', 'SPEED_MA', 'SPEED_STD', 'RPM_MA', 'RPM_STD', 'THROTTLE_MA', 'THROTTLE_STD', 'HARSH_ACCEL', 'HARSH_BRAKE', 'HIGH_RPM', 'HIGH_SPEED', 'IDLING', 'DRIVING_STYLE']

최종 병합 결과:
  기존 데이터:       60439건
  난폭운전 데이터:   6000건
  합계:              66439건

전체 라벨 분포:
DRIVING_STYLE
safe          35904
aggressive    19284
normal        11251
Name: count, dtype: int64

비율:
DRIVING_STYLE
safe          0.540
aggressive    0.290
normal        0.169
Name: proportion, dtype: float64

저장 완료: obd_final.csv
